# Part teòrica

**Variables:** Files i columnes (caselles del taulell)

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** k^(mxn) on m son les files, n les columnes del taulell y k les lletres de l'alfabet

**Estratègia de millora:** Fer ús d'una heurística per a decidir quina assignació fer a continuació. Per exemple triar una paraula amb el major nombre de restriccions respecte a altres no asignades, ja que si té un major impacte sobre la resta pot portar a acabar abans


# Codi

## Carregar biblioteques

In [81]:
import numpy as np

# **Exercici 1**

In [82]:
data = []
words = []

with open('crossword_CB_v3.txt', 'r') as file:
    for line in file.readlines():
        elements = line.strip().split()
        data.append(elements)
        
taulell = np.array(data)

with open('diccionariAA.txt', 'r') as file:
    for word in file.readlines():
        words.append(word.strip())

diccionary = np.array(words)

print(taulell)


[['0' '0' '0' '0' '0' '0']
 ['0' '#' '#' '0' '#' '0']
 ['0' '#' '0' '0' '0' '0']
 ['0' '#' '#' '0' '#' '0']
 ['#' '0' '0' '0' '0' '0']
 ['0' '0' '0' '0' '#' '#']
 ['0' '0' '#' '#' '#' '#']]


In [83]:
#variable = [[[pos_inicial], len, v/h]] vertical = 1, horitzontal = 0

def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [84]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [85]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [86]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

In [87]:
def interseccions(variables):
    positions = []
    intersections = []
    for variable in variables:
        for pos in calculaPosicionsVariable(variable):
            if pos not in positions:
                positions.append(pos)
            elif pos not in intersections:
                intersections.append(pos)
                
    
    return(intersections)

In [88]:
def isValid(word, variable, assigned_variables, taulell):
    if len(word) != variable[1]:
        return False

    for assigned_variable in assigned_variables:
        if word == assigned_variable[1]:
            return False
    
    aux_taulell = taulell
    
    for var, word in assigned_variables:
        positions = calculaPosicionsVariable(var)
        for i, j in positions:
            aux_taulell[i][j] = word[i - var[0][0] if var[2] == 1 else j - var[0][1]]

    for position, letter in zip(calculaPosicionsVariable(variable), word):
        i, j = position
        if aux_taulell[i][j] != '0' and aux_taulell[i][j] != letter:
            return False

    return True

```
Funcio Backtracking(LVA,LVNA,R,D)
    Si (LVNA és buida) llavors Retornar(LVA) fSi
    Var=Cap(LVNA);
    Per a cada (valor del Domini(Var, D) que podem assignar a Var) fer
        Si (SatisfaRestriccions([Var valor],LVA,R)) llavors
            Res=Backtracking(Insertar([Var, valor],LVA),Cua(LVNA),R,D);
            Si (Res és una solució completa) llavors
                Retornar(Res);
            Fsi
        Fsi
    Fper
    Retornar(Falla)
FFuncio

```

In [89]:
def backtracking(assigned_variables, variables, taulell, diccionary):
    if not variables:
        return assigned_variables

    var = variables[0]  

    for word in diccionary:
        if isValid(word, var, assigned_variables, taulell):
            assigned_variables.append([var, word])
            result = backtracking(assigned_variables, variables[1:], taulell, diccionary)
            
            if result: 
                return result
            
            variables.append(assigned_variables.pop()[0]) 
            

    return None

assigned_variables = []
variables = []
cercaVariables(taulell, variables)

res = backtracking(assigned_variables, variables, taulell, diccionary)

print(res)

None


In [90]:
def printSolution(assigned_variables, taulell):
    for variable, word in assigned_variables:
        positions = calculaPosicionsVariable(variable)
        for i, j in positions:
            taulell[i][j] = word[i - variable[0][0] if variable[2] == 1 else j - variable[0][1]]

    for row in taulell:
        print(" ".join(row))



printSolution(assigned_variables, taulell)
taulell

A C A T A R
0 # # 0 # 0
0 # A L G A
0 # # 0 # 0
# P I P A L
A B A T # #
L A # # # #


array([['A', 'C', 'A', 'T', 'A', 'R'],
       ['0', '#', '#', '0', '#', '0'],
       ['0', '#', 'A', 'L', 'G', 'A'],
       ['0', '#', '#', '0', '#', '0'],
       ['#', 'P', 'I', 'P', 'A', 'L'],
       ['A', 'B', 'A', 'T', '#', '#'],
       ['L', 'A', '#', '#', '#', '#']], dtype='<U1')